La variable cible est Class : 1 = transaction frauduleuse, 0 = transaction normale (selon la documentation du jeu de données).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED = 42
df = pd.read_csv("creditcard.csv")

In [2]:
print(df.columns)

Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='str')


In [3]:
print("Valeurs manquantes :",df.isna().sum().sum())
print("Doublons :",df.duplicated().sum())
print(df["Class"].value_counts())

print("Taux de fraude :", round(df["Class"].mean()*100,3),"%")

Valeurs manquantes : 0
Doublons : 1081
Class
0    284315
1       492
Name: count, dtype: int64
Taux de fraude : 0.173 %


In [4]:
doublons = df[df.duplicated()]
print(doublons["Class"].value_counts())

Class
0    1062
1      19
Name: count, dtype: int64


In [5]:
print("Avant :", df.shape)

df=df.drop_duplicates()
print("Après :", df.shape)
print(df["Class"].value_counts())


Avant : (284807, 31)
Après : (283726, 31)
Class
0    283253
1       473
Name: count, dtype: int64


On supprime les 1 081 doublons : une ligne identique dans le train et le test fausserait l’évaluation. Cela retire environ 19 fraudes, il en reste 473.

In [6]:
print(df.groupby("Class")["Amount"].describe())

          count        mean         std  min   25%    50%     75%       max
Class                                                                      
0      283253.0   88.413575  250.379023  0.0  5.67  22.00   77.46  25691.16
1         473.0  123.871860  260.211041  0.0  1.00   9.82  105.89   2125.87


Les fraudes ont une médiane plus faible (9,82 € contre 22 €) mais une moyenne plus élevée (124 € contre 88 €) : la plupart sont de petits montants, mais quelques-unes sont importantes. 25 % des fraudes font 1 € ou moins, ce qui correspond à des tests de carte. Toutes les fraudes n’ont donc pas le même coût, ce qui comptera pour la décision finale.

In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

def report(nom, y_vrai, y_pred) :
    precision = precision_score(y_vrai, y_pred, zero_division=0)
    rappel = recall_score(y_vrai, y_pred)
    f1 = f1_score(y_vrai, y_pred)
    
    print("Modèle :", nom)
    print("Précision :", round(precision, 3))
    print("Rappel :", round(rappel,3))
    print("F1 :", round(f1,3))
    print(confusion_matrix(y_vrai, y_pred))


In [8]:
y_naif = np.zeros(len(df))
report("Naïf", df["Class"], y_naif)

Modèle : Naïf
Précision : 0.0
Rappel : 0.0
F1 : 0.0
[[283253      0]
 [   473      0]]


In [9]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(df["Class"], y_naif)
print("Accuracy :", round(accuracy,4))

Accuracy : 0.9983


Modèle naïf. Ce modèle prédit « normal » pour toutes les transactions. La matrice de confusion le confirme : la colonne « fraude prédite » ne contient que des zéros. Les 473 fraudes sont toutes manquées (FN = 473, VP = 0), donc la précision, le rappel et le F1 sont nuls.

Pourtant, son accuracy est de 99,8 % (voir cellule ci-dessus). Quand la fraude est rare, l'accuracy est donc trompeuse : un modèle inutile obtient un score presque parfait. Dans la suite, j'utilise la précision, le rappel et le F1, et ce modèle naïf sert de référence (baseline).

## Séparation des données

In [11]:
X = df.drop(columns="Class")
y=df["Class"]

print("X :", X.shape)
print("y :", y.shape)



X : (283726, 30)
y : (283726,)


## Train/Test

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, stratify=y, random_state=SEED)

print("Train :", len(y_train), "transactions,", y_train.sum(), "fraudes")
print("Test :", len(y_test), "transactions,", y_test.sum(), "fraudes")

Train : 226980 transactions, 378 fraudes
Test : 56746 transactions, 95 fraudes


Séparation train/ test : On sépare les données en deux parties : 80 % pour entraîner les modèles (train) et 20 % pour les évaluer à la fin (test).

On utilise stratify=y : comme les fraudes sont très rares (0,17 %), un découpage purement aléatoire pourrait mettre, par hasard, trop peu ou trop de fraudes dans le test. Avec stratify, le taux de fraude reste le même dans le train et dans le test. Le paramètre random_state permet d'obtenir le même découpage à chaque exécution.



## Régression logistique

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

In [14]:
print(X_train_sc.mean().round(2), X_train_sc.std().round(2))

-0.0 1.0
